In [2]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
import utils_2Q_gate_zp as ut
ut.set_fig_font() ### Set various sizes in plotting
import scipy as sp
from joblib import Parallel, delayed
import itertools
from qutip.qip.operations import rz, cz_gate, cnot, rx, hadamard_transform, swap
import pandas as pd

### Get fidelity for input params (pick=True)

In [2]:
truc1, truc_tot, charge_pick = 300, 1000, True
truc_full = 1000

folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')

truc_list = np.arange(truc_full)
hspace_full = hspace_full[:truc_full]
eval_tot = eval_tot[:truc_full]
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
logi_state = ['0-0', '0-2', '2-0', '2-2']
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]

num_cpus = 16
x0_vec = np.array([
 [182.99766, 0.007776, 0.01307, -4.42509061, -4.37463037],
])
n_job = 100
c_op_list = []
H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [H0_full, [n_theta1_dress, ut.drive_gauss_A] ]

In [7]:
arg_full = [H_drive_full, W_20_50, num_cpus, c_op_list, logi_idx_full ]
f_full = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_full)
                                            for args_indep in x0_vec[:, :3])
print(f' fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())

 fidelity (dim=1000,True) = [-4.2781217]


In [ ]:
# len_part = 190
# hspace_part = hspace_full[:len_part]
# index_part = np.arange(len_part)
# H0_part = ut.truncate_2( H0_full, index_part )
# n_theta1_part = ut.truncate_2(n_theta1_dress, index_part)
# logi_idx_part = [hspace_part.index(i) for i in logi_state]
# H_drive_part = [H0_part, [n_theta1_part, ut.drive_gauss_A] ]

# arg_part = [H_drive_part, W_20_50, num_cpus, c_op_list, logi_idx_part ]
# f_part = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_part)
#                                             for args_indep in x0_vec[:, :3])
# print(f' fidelity (dim={len_part},{charge_pick}) =', np.round(f_part, 8).tolist())

 fidelity (dim=190,True) = [-0.29474067, -2.21903731, -3.9384408]


### Get fidelity for input params (pick=False)

In [3]:
truc1, truc_tot, charge_pick = 300, 2000, False
truc_full = 2000
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')

truc_list = np.arange(truc_full)
hspace_full = hspace_full[:truc_full]
eval_tot = eval_tot[:truc_full]
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]

H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [H0_full, [n_theta1_dress, ut.drive_gauss_A] ]

In [ ]:
arg_full = [H_drive_full, W_20_50, num_cpus, c_op_list, logi_idx_full ]
f_full = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_full)
                                            for args_indep in x0_vec[:, :3])
print(f' fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())

 fidelity (dim=2000,False) = [-4.14998241]


: 

In [ ]:
# len_part = 500
# hspace_part = hspace_full[:len_part]
# index_part = np.arange(len_part)
# H0_part = ut.truncate_2( H0_full, index_part )
# n_theta1_part = ut.truncate_2(n_theta1_dress, index_part)
# logi_idx_part = [hspace_part.index(i) for i in logi_state]
# H_drive_part = [H0_part, [n_theta1_part, ut.drive_gauss_A] ]

# arg_part = [H_drive_part, W_20_50, num_cpus, c_op_list, logi_idx_part ]
# f_part = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_part)
#                                             for args_indep in x0_vec)
# print(f' fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())
# print(f' fidelity (dim={len_part},{charge_pick}) =', np.round(f_part, 8).tolist())

 fidelity (dim=500,False) = [-0.2947592, -2.2188636, -3.94444026]


In [ ]:
if len(c_op_list) == 0 and H0.isoper:
    # calculate propagator for the wave function

    N = H0.shape[0]
    dims = H0.dims

    if parallel:
        unitary_mode = 'single'
        u = np.zeros([N, N, len(tlist)], dtype=complex)
        output = parallel_map(_parallel_sesolve, range(N),
                                task_args=(N, H, tlist, args, options),
                                progress_bar=progress_bar, num_cpus=num_cpus)
        for n in range(N):
            for k, t in enumerate(tlist):
                u[:, n, k] = output[n].states[k].full().T    
else:
    # calculate the propagator for the vector representation of the
    # density matrix (a superoperator propagator)
    unitary_mode = 'single'
    N = H0.shape[0]
    dims = [H0.dims, H0.dims]

    u = np.zeros([N * N, N * N, len(tlist)], dtype=complex)

    if parallel:
        output = parallel_map(_parallel_mesolve, range(N * N),
                                task_args=(
                                    N, H, tlist, c_op_list, args, options),
                                task_kwargs={"dims": H0.dims},
                                progress_bar=progress_bar, num_cpus=num_cpus)
        for n in range(N * N):
            for k, t in enumerate(tlist):
                u[:, n, k] = mat2vec(output[n].states[k].full()).T

def _parallel_sesolve(n, N, H, tlist, args, options):
    psi0 = basis(N, n)
    output = sesolve(H, psi0, tlist, [], args, options, _safe_mode=False)
    return output                